# Uber Fare Prediction — Linear Regression vs Random Forest Regression

This notebook preprocesses the Uber Fares dataset, handles missing values and outliers, explores correlations, trains Linear Regression and Random Forest Regression models, and compares R², RMSE, and MAE.

In [ ]:
# Install required libraries (Google Colab)
!pip -q install pandas numpy matplotlib seaborn scikit-learn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
  warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)


In [ ]:
# Upload the Kaggle CSV file in Colab
from google.colab import files

uploaded = files.upload()


In [ ]:
# Load dataset
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print('Shape:', df.shape)
display(df.head())


## 1. Data Preprocessing

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower()

print(df.columns.tolist())
  
# Convert pickup_datetime to datetime
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')

# Remove invalid rows
df = df.dropna(subset=['pickup_datetime', 'fare_amount', 'pickup_longitude',
                       'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
                       'passenger_count'])

# Remove invalid fare values and passenger counts
df = df[df['fare_amount'] > 0]
df = df[df['passenger_count'] > 0]

print('Shape after basic cleaning:', df.shape)


In [ ]:
# Check missing values
print(df.isnull().sum())

# Basic statistics
display(df.describe())


## 2. Feature Engineering

In [ ]:
# Extract useful time features
df['year'] = df['pickup_datetime'].dt.year
df['month'] = df['pickup_datetime'].dt.month
df['day'] = df['pickup_datetime'].dt.day
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek

# Haversine distance between pickup and drop-off points
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine_distance(
    df['pickup_latitude'],
    df['pickup_longitude'],
    df['dropoff_latitude'],
    df['dropoff_longitude']
)

display(df.head())


## 3. Outlier Detection and Removal

In [ ]:
# Keep geographically meaningful NYC coordinates for this dataset
df = df[
    df['pickup_latitude'].between(40.0, 42.0) &
    df['dropoff_latitude'].between(40.0, 42.0) &
    df['pickup_longitude'].between(-75.0, -72.0) &
    df['dropoff_longitude'].between(-75.0, -72.0)
]

# Remove zero-distance trips and extreme distances
df = df[(df['distance_km'] > 0) & (df['distance_km'] <= 100)]

# IQR-based outlier removal for fare amount
Q1 = df['fare_amount'].quantile(0.25)
Q3 = df['fare_amount'].quantile(0.75)
IQR = Q3 - Q1

lower = max(0, Q1 - 1.5 * IQR)
upper = Q3 + 1.5 * IQR

df = df[df['fare_amount'].between(lower, upper)]

print('Fare lower bound:', lower)
print('Fare upper bound:', upper)
print('Shape after outlier removal:', df.shape)


In [ ]:
# Visualize fare outliers before/after cleaning
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['fare_amount'])
plt.title('Fare Amount After Outlier Removal')
plt.show()


## 4. Correlation Analysis

In [ ]:
corr_cols = [
    'fare_amount', 'distance_km', 'passenger_count',
    'year', 'month', 'day', 'hour', 'day_of_week',
    'pickup_longitude', 'pickup_latitude',
    'dropoff_longitude', 'dropoff_latitude'
]

corr = df[corr_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


## 5. Prepare Features and Target

In [ ]:
features = [
    'pickup_longitude', 'pickup_latitude',
    'dropoff_longitude', 'dropoff_latitude',
    'passenger_count', 'year', 'month', 'day',
    'hour', 'day_of_week', 'distance_km'
]

X = df[features]
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print('Training samples:', len(X_train))
print('Testing samples :', len(X_test))


## 6. Linear Regression

In [ ]:
# Scale features for Linear Regression
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

linear_pred = linear_model.predict(X_test_scaled)

linear_r2 = r2_score(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_mae = mean_absolute_error(y_test, linear_pred)

print('Linear Regression')
print('R2   :', linear_r2)
print('RMSE :', linear_rmse)
print('MAE  :', linear_mae)


## 7. Random Forest Regression

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)

print('Random Forest Regression')
print('R2   :', rf_r2)
print('RMSE :', rf_rmse)
print('MAE  :', rf_mae)


## 8. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regression'],
    'R2': [linear_r2, rf_r2],
    'RMSE': [linear_rmse, rf_rmse],
    'MAE': [linear_mae, rf_mae]
})

display(results)


In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.barplot(data=results, x='Model', y='R2', ax=axes[0])
axes[0].set_title('R² Comparison')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=results, x='Model', y='RMSE', ax=axes[1])
axes[1].set_title('RMSE Comparison')
axes[1].tick_params(axis='x', rotation=20)

sns.barplot(data=results, x='Model', y='MAE', ax=axes[2])
axes[2].set_title('MAE Comparison')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


## 9. Actual vs Predicted Fare

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, rf_pred, alpha=0.25)
plt.xlabel('Actual Fare')
plt.ylabel('Predicted Fare')
plt.title('Random Forest: Actual vs Predicted Fare')
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle='--'
)
plt.show()


## 10. Feature Importance — Random Forest

In [ ]:
importance = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

display(importance)

plt.figure(figsize=(9, 5))
sns.barplot(data=importance, x='Importance', y='Feature')
plt.title('Random Forest Feature Importance')
plt.show()


## 11. Predict Fare for a New Ride

In [ ]:
# Example ride
# Replace these values with the pickup/drop-off coordinates and ride details.
new_ride = pd.DataFrame({
    'pickup_longitude': [-73.985],
    'pickup_latitude': [40.748],
    'dropoff_longitude': [-73.985],
    'dropoff_latitude': [40.758],
    'passenger_count': [1],
    'year': [2015],
    'month': [6],
    'day': [15],
    'hour': [18],
    'day_of_week': [0],
    'distance_km': [1.2]
})

# Linear Regression prediction
new_ride_scaled = scaler.transform(new_ride)
linear_fare = linear_model.predict(new_ride_scaled)[0]

# Random Forest prediction
rf_fare = rf_model.predict(new_ride)[0]

print('Predicted fare using Linear Regression : $', round(linear_fare, 2))
print('Predicted fare using Random Forest     : $', round(rf_fare, 2))
